# Modul 5: Pemodelan SARIMA Metodologi Box-Jenkins
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 5 (Minggu 5):** Menerapkan siklus Box-Jenkins (Identifikasi, Estimasi MLE, Diagnostik Residual Ljung-Box, & Peramalan dengan Confidence Interval 95%) pada beban berkala musiman.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

warnings.filterwarnings('ignore')
plt.rcParams['axes.grid'] = True

## 1. Dataset Beban Gardu Distribusi (14 Hari, Periode Musiman $s = 24$)

In [ ]:
rng = np.random.default_rng(55)
n_hours = 24 * 14
idx = pd.date_range('2026-03-01', periods=n_hours, freq='h')
t = np.arange(n_hours)
hour = idx.hour

load = (
    300 + 0.12 * t
    + 70 * np.sin(2 * np.pi * (hour - 6) / 24)
    + 35 * np.exp(-((hour - 19) ** 2) / 6)
    + rng.normal(0, 6.0, n_hours)
)
s = pd.Series(load, index=idx, name='substation_mw')
train, test = s.iloc[:-48], s.iloc[-48:]

## 2. Estimasi SARIMA(1,1,1)(1,1,0)[24] & Uji Diagnostik Ljung-Box

In [ ]:
model = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 0, 24))
res = model.fit(disp=False)
print(res.summary())

# Diagnostik Residual: Ljung-Box Q-test
lb_test = acorr_ljungbox(res.resid.iloc[25:], lags=[12, 24], return_df=True)
print("\n=== Uji Diagnostik Ljung-Box (H0: Residual adalah White Noise) ===")
print(lb_test)

res.plot_diagnostics(figsize=(12, 7))
plt.tight_layout()
plt.show()

## 3. Peramalan 48 Jam ke Depan & Pita Keyakinan 95%

In [ ]:
fc_obj = res.get_forecast(steps=len(test))
fc_mean = fc_obj.predicted_mean
conf_int = fc_obj.conf_int(alpha=0.05)

rmse = np.sqrt(np.mean((test.values - fc_mean.values) ** 2))
mape = np.mean(np.abs((test.values - fc_mean.values) / test.values)) * 100
print(f"Evaluasi Out-of-Sample (48 jam) -> RMSE: {rmse:.2f} MW | MAPE: {mape:.2f}%")

plt.figure(figsize=(12, 5))
plt.plot(train.index[-72:], train.iloc[-72:], label='Histori Train', color='gray')
plt.plot(test.index, test, label='Aktual Test', color='black', lw=2)
plt.plot(test.index, fc_mean, label='Forecast SARIMA', color='crimson', lw=2)
plt.fill_between(test.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1], color='crimson', alpha=0.2, label='95% CI')
plt.title('Peramalan Beban Gardu Distribusi dengan SARIMA(1,1,1)(1,1,0)[24]')
plt.ylabel('Beban (MW)')
plt.legend()
plt.show()

## 4. Tugas & Pertanyaan Analisis (C4–C6)
1. **[C4]** Periksa nilai `lb_pvalue` pada lag 12 dan 24. Apakah asumsi *white noise* terpenuhi ($p > 0.05$)?
2. **[C5]** Bandingkan AIC dan RMSE antara `SARIMA(1,1,1)(1,1,0)[24]` vs. `SARIMA(1,1,1)(0,1,1)[24]`.